# Reto 06 — Archivos en lote y base de datos

**Caso:** la tienda te manda las ventas en **un CSV por mes**, y cada mes llega uno nuevo.
Nadie va a abrir 12 archivos a mano, ni 60 el ano que viene.

Escribe un proceso que consolide la carpeta entera, la cargue a SQLite y responda por SQL.
La regla de este modulo: **el codigo no puede saber cuantos archivos hay.**

In [ ]:
from pathlib import Path
import pandas as pd
import sqlite3

CARPETA = Path("../../_RECURSOS/datasets/ventas_mensuales")
assert CARPETA.exists(), f"No encuentro {CARPETA.resolve()}"

## 1 — Listar la carpeta

Lista todos los `.csv` de `CARPETA`, ordenados por nombre, en la variable `archivos`.

Prohibido escribir los 12 nombres a mano. Si el mes que viene aparece
`ventas_2026-01.csv`, tu codigo debe recogerlo solo.

In [ ]:
archivos = ...


# --- verificacion (no modificar) ---
assert len(archivos) == 12, f"Esperaba 12 archivos, encontre {len(archivos)}"
assert all(isinstance(a, Path) for a in archivos)
for a in archivos:
    print(a.name)

## 2 — Consolidar

Lee los 12 y unelos en un solo DataFrame `todo`. Agrega una columna
`archivo_origen` con el nombre del archivo del que salio cada fila.

Esa columna parece de adorno hasta el dia que un mes venga mal y tengas que
saber cual reprocesar. Siempre se agrega.

Pista: acumula en una lista y haz **un solo** `pd.concat` al final. Concatenar dentro
del bucle copia el DataFrame entero en cada vuelta.

In [ ]:
todo = ...


# --- verificacion (no modificar) ---
assert len(todo) == 491, f"Esperaba 491 filas, salieron {len(todo)}"
assert "archivo_origen" in todo.columns
assert todo["archivo_origen"].nunique() == 12
print("OK 2 —", todo.shape)

## 3 — Limpiar

Aplica lo mismo del reto 04, pero ahora **como una funcion** `limpiar(df)`.
Una funcion, porque este proceso se va a correr cada mes.

Debe: quitar duplicados, normalizar `canal`, convertir `fecha` a datetime
(`dayfirst=True`) y recalcular los `importe` nulos.

In [ ]:
def limpiar(df):
    """TODO"""
    ...


limpio = limpiar(todo)

# --- verificacion (no modificar) ---
assert len(limpio) == 482, f"Esperaba 482 tras deduplicar, salieron {len(limpio)}"
assert sorted(limpio["canal"].unique()) == ["Telefono", "Tienda", "Web"]
assert limpio["fecha"].dtype.kind == "M"
assert limpio["importe"].isna().sum() == 0
print("OK 3 — limpio:", limpio.shape)

## 4 — Cargar a SQLite

Crea `tienda.db` al lado de este notebook y escribe `limpio` en la tabla `ventas`.

Usa `if_exists="replace"`. Piensa por que: que pasaria con `"append"` si corres
la celda dos veces?

In [ ]:
# TODO


# --- verificacion (no modificar) ---
con = sqlite3.connect("tienda.db")
n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", con)["n"][0]
assert n == 482, f"La tabla tiene {n} filas"
print("OK 4 — tabla ventas con", n, "filas")

## 5 — Responder por SQL

Responde estas 3 con `pd.read_sql`, **no** con pandas. El punto es que veas que el
mismo `groupby` se escribe en los dos idiomas.

1. Top 5 ciudades por importe total
2. Importe por canal y por mes (mes como `strftime('%Y-%m', fecha)`)
3. Los 3 clientes que mas compraron

In [ ]:
# TODO: consulta 1

In [ ]:
# TODO: consulta 2

In [ ]:
# TODO: consulta 3

## 6 — Escribir el reporte a disco

Exporta el resultado de la consulta 1 a `reporte_ciudades.txt` como texto legible,
usando `with open(...)` y `pathlib`.

In [ ]:
# TODO


# --- verificacion (no modificar) ---
assert Path("reporte_ciudades.txt").exists()
print(Path("reporte_ciudades.txt").read_text(encoding="utf-8"))
con.close()

## 7 (opcional, +2 puntos) — Idempotencia

Convierte todo lo anterior en una funcion `procesar_carpeta(carpeta, db)` que se pueda
correr N veces seguidas y siempre deje la base igual. Demuestralo: llamala dos veces
y verifica que la tabla sigue teniendo 482 filas.

In [ ]:
# TODO

---
## Criterios de aceptacion

Antes de entregar, verifica que se cumpla **todo** lo de abajo.

- [ ] *Restart & Run All* corre limpio y todos los `OK` salen
- [ ] En ningun lado hay un nombre de archivo escrito a mano
- [ ] Hay **un solo** `pd.concat`, fuera del bucle
- [ ] Todas las rutas usan `pathlib.Path`, no concatenacion de strings con `\`
- [ ] La limpieza esta encapsulada en la funcion `limpiar(df)`
- [ ] Las 3 consultas son SQL de verdad, no `groupby` de pandas disfrazado
- [ ] `tienda.db` y `reporte_ciudades.txt` quedan generados